# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset package using the `mlcroissant` library. All references to dataset entities strictly follow their Croissant `@id` identifiers for clarity and reproducibility.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Install mlcroissant if it's not already available
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
metadata_dict = json.loads(metadata_obj.to_json_string())

print(f"Dataset Name: {metadata_dict['name']}")
print(f"Description: {metadata_dict['description']}")
print(f"Identifier: {metadata_dict['identifier']}")
print(f"Keywords: {metadata_dict['keywords']}")
print(f"License: {metadata_dict['license']}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s, as defined in Croissant Schema.

In [ ]:
# Retrieve Croissant schema for record sets and fields

# Get all record sets and fields by their @id
croissant_schema = dataset.metadata.to_json_dict()
record_sets = croissant_schema.get('recordSet', [])

def get_entity(entity_list, entity_type):
    return [item['@id'] for item in entity_list if item.get('@type')==entity_type]

print("Available record sets:")
for rec in record_sets:
    print('- RecordSet @id:', rec['@id'])

# List fields in each record set
for rec in record_sets:
    print(f"\nFields for RecordSet {rec['@id']}:")
    fields = rec.get('field', [])
    for f in fields:
        fid = f['@id'] if isinstance(f, dict) else f
        print(f"  Field @id: {fid}")

## 3. Data Extraction
Load data from each record set using their Croissant `@id`, creating a DataFrame for each. Use field `@id` for column selection.

In [ ]:
# Extract records from all record sets into pandas DataFrames
dataframes = {}

record_set_ids = [rec['@id'] for rec in record_sets]
print("Extracting from record sets:", record_set_ids)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns in record set {record_set_id}:\n", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Explore, filter, and transform data using column `@id`.

Choose a numeric field to process, normalize, and group by another field (all references by `@id`). Remove outliers and transform values as example.

In [ ]:
# Example: Pick first record set and numeric field
# Inspect column @ids from previous extraction

# Set record set and field @id
example_record_set_id = record_set_ids[0]
df = dataframes[example_record_set_id]
print(f"Columns in {example_record_set_id}: {df.columns.tolist()}")

# Assume an age column exists, find its @id
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower() or 'height' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    numeric_field_id = df.select_dtypes(include='number').columns[0]

print(f"Using numeric field for EDA: {numeric_field_id}")

# Filter: values > threshold
threshold = 20
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field (pick one with few unique values)
group_field_id = None
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() < 5 and col != numeric_field_id:
        group_field_id = col
        break

if group_field_id:
    print(f"Grouping by {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id).mean()[[numeric_field_id, f"{numeric_field_id}_normalized"]]
    print("Grouped means:")
    print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions or relationships between fields. Field and group are referenced by `@id`.

In [ ]:
# Visualize normalized numeric field distribution
if numeric_field_id:
    plt.figure(figsize=(7,4))
    filtered_df[f"{numeric_field_id}_normalized"].plot(kind='hist', bins=10, alpha=0.7)
    plt.title(f"Distribution of Normalized {numeric_field_id} in {example_record_set_id}")
    plt.xlabel("Normalized Value")
    plt.ylabel("Frequency")
    plt.show()

# If grouping field exists, plot group means
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean()
    grouped_df[f"{numeric_field_id}_normalized"].plot(kind='bar', figsize=(7,4))
    plt.title(f"Mean Normalized {numeric_field_id} by {group_field_id} (RecordSet: {example_record_set_id})")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean Normalized {numeric_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, inspecting, filtering, normalizing, grouping, and visualizing the FAIR^2 dataset using Croissant `@id` references for record sets and fields.

- **All operations referenced entities by their `@id` for reproducibility.**
- **Exploratory analysis illustrated small cohort limitations, column normalization, and potential group-wise patterns.**
- **Visualizations highlighted the distribution and grouping of a key numeric field.**

This approach can be adapted for other Croissant datasets and refined as more fields and relationships are identified.